# Resource Allocation: Cores, Memory, and Executor Sizing Formula

**Module**: 12_advanced_topics  
**Notebook**: 04_resource_allocation.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The exact mathematical formula for cluster resource allocation: sizing executor memory, cores, instances, and OS buffers.
- **Why does this topic exist?**: Suboptimal resource requests waste thousands of cloud dollars or cause JVM thrashing.
- **What problem does it solve?**: Choosing between 100 tiny 1-core executors vs 2 massive 50-core executors.
- **Where is it used in real-world Data Engineering?**: Sizing multi-node clusters on AWS EMR, Databricks, and GCP Dataproc.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Executor sizing, cluster resources, memory overhead |
| **Required depth** | Working knowledge: can size executors and explain the trade-offs |
| **Data Engineer** | Choose executor count, cores, and memory for a workload. |
| **Senior Data Engineer** | Balance parallelism, memory, and scheduling overhead on a shared cluster. |
| **Spark specialist** | Reason about resource fragmentation, dynamic allocation, and queue capacity. |
| **Next topic** | `12_advanced_topics/05_spark_connect.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Executor sizing is a trade-off between parallelism, memory per task, and scheduling overhead. Many small
executors improve fault isolation and scheduling flexibility but add overhead; a few large executors reduce
overhead but increase the blast radius of a failure and can waste resources. There is no universal formula; the
right answer depends on the workload, cluster, and cluster manager.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Executor cores** | Concurrent tasks per executor | `spark.executor.cores` |
| **Executor memory** | Heap per executor | `spark.executor.memory` |
| **Memory overhead** | Non-heap memory | `spark.executor.memoryOverhead` |
| **Executor count** | Number of executors | Fixed or dynamic |
| **Cores per node** | Node capacity | Leave room for OS and daemons |
| **Resource fragmentation** | Unused capacity | Large containers may not fit available slots |

### How it works, step by step

1. The driver requests executors with a core and memory shape.
2. The cluster manager places containers on nodes with enough free resources.
3. Each executor runs one task per core concurrently.
4. Memory per task depends on the executor heap and the number of concurrent tasks.
5. Dynamic allocation adds or removes executors based on backlog.
6. The job's parallelism is bounded by total cores and partition count.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. The Golden Executor Sizing Formula
Given a 10-node cluster (Each node: 16 cores, 64 GB RAM):
1. **OS / Daemon Allocation**: Reserve 1 core and 4 GB RAM per node for OS, NodeManager/Worker.
   - Usable per node: 15 cores, 60 GB RAM.
2. **Cores per Executor**: Set to **5 cores** (Optimal for HDFS throughput & GC).
   - Executors per node: $15 / 5 = 3$ executors. Total executors across 10 nodes = $30$.
3. **Memory per Executor**: $60	ext{GB} / 3 = 20	ext{GB}$ total process memory.
   - Overhead (10%): $20	ext{GB} 	imes 0.10 = 2	ext{GB}$.
   - Heap (`spark.executor.memory`): $20 - 2 = \mathbf{18	ext{GB}}$.
4. Reserve 1 executor for the ApplicationMaster / Driver.
   - Final: `--num-executors 29 --executor-cores 5 --executor-memory 18g`.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Resource_Alloc_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Sizing calculation function
def calculate_cluster_sizing(nodes, cores_per_node, ram_gb_per_node):
    usable_cores = cores_per_node - 1
    usable_ram = ram_gb_per_node - 4
    cores_per_exec = 5
    execs_per_node = usable_cores // cores_per_exec
    total_execs = (execs_per_node * nodes) - 1 # reserve 1 for Driver
    ram_per_exec = usable_ram / execs_per_node
    overhead = max(0.384, ram_per_exec * 0.10)
    heap = ram_per_exec - overhead
    return {
        "total_executors": total_execs,
        "cores_per_executor": cores_per_exec,
        "heap_memory_gb": round(heap, 1),
        "overhead_memory_gb": round(overhead, 1)
    }

print("Recommended sizing for 6-node cluster (32 cores, 128 GB RAM each):")
print(calculate_cluster_sizing(6, 32, 128))

---

## Experiment: Validating Our Lab Sizing

- **1. What are we testing?**: Verify resource allocation in our 2-worker Docker cluster.
- **2. What do we expect?**: 2 workers x 2 cores x 4GB = 4 cores, 8GB total.

In [ ]:
print('Lab Cluster: 2 workers, 2 cores each = 4 total executor cores.')

**3. What happened?**: Matched docker-compose.yml parameters.  
**4. Why did it happen?**: Sized for lightweight local development.  
**5. Practical Engineering Lesson**: Always calculate executor quotas based on physical node limits.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Right-sizing a shared cluster

- **Industry**: Enterprise platform
- **Scenario**: A cluster runs a mix of small and large jobs with fixed executor settings.
- **Problem**: Large executors fragment the cluster and small jobs wait for capacity.
- **Why this concept is relevant**: Smaller executors improve packing and scheduling flexibility for mixed workloads.
- **Implementation idea**: Compare queue wait time and utilization with different executor shapes.
- **Expected result**: Higher utilization without violating SLAs.
- **Alternative**: Dedicated queues per workload type.
- **Trade-offs**: Dedicated queues isolate workloads but reduce sharing efficiency.

### Use Case 2: Memory-heavy aggregation

- **Industry**: Retail
- **Scenario**: A large aggregation spills heavily with four-core executors.
- **Problem**: Each core shares the executor heap, so concurrency reduces memory per task.
- **Why this concept is relevant**: Fewer cores per executor can give each task more memory and reduce spill.
- **Implementation idea**: Test executor shapes and compare spill, GC, and runtime.
- **Expected result**: A shape that balances concurrency and memory.
- **Alternative**: Increase executor memory.
- **Trade-offs**: More memory helps but may increase cost and GC pause time.

---

## Business Value

- Higher cluster utilization.
- Lower queue wait times.
- Fewer spill and OOM incidents.
- Predictable cost per workload.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Leave resources for the OS and cluster daemons.
- Do not copy a universal cores-per-executor formula.
- Measure spill and GC when changing executor shape.
- Account for driver resources and queue capacity.
- Use dynamic allocation when workload size varies.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare executor shapes

**Steps**

1. Run the same job with 4 executors x 4 cores and 8 executors x 2 cores (same total cores).
2. Record runtime, spill, GC, and task duration.
3. Choose the shape that best fits the workload.

**What to observe**: The total cores may be the same, but memory per task and scheduling overhead differ.

**Healthy result**: The chosen shape improves the bottleneck without wasting capacity.

**Problematic result**: More executors do not help because the partition count is too low.

**Engineering takeaway**: Executor shape and partition count must be tuned together.

---

## Trade-offs and Alternatives

### Advantages

- Direct control of parallelism
- Tunable for workload shape
- Works across cluster managers

### Limitations

- No universal formula
- Fragmentation
- Large executors increase failure blast radius

### When to use it

- Capacity planning
- Performance tuning
- Shared clusters

### When not to use it

- Oversized executors
- Too many tiny executors
- Ignoring overhead

### Alternatives inside Spark

- Executor cores/memory
- Dynamic allocation
- Driver resources

### Alternatives outside Spark

- Cluster autoscaling
- Queue policies
- Node sizing

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Core Sizing Rule
**Problem Statement**: What is the recommended core count per executor?  
**Input / Context**: Cores

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Calculate 4-Node Cluster
**Problem Statement**: Calculate sizing for 4 nodes (16 cores, 64 GB).  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tiny vs Fat Executor Trade-off
**Problem Statement**: Explain drawbacks of 1-core vs 16-core executors.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The cluster with free memory but no free slots

**Scenario**: A job waits even though the cluster has free memory.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the executor container shape fits the free slots.
2. Look at cores per node and fragmentation.
3. Consider smaller executors or dynamic allocation.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Core Sizing Rule):
```python
print('Between 3 and 5 cores per executor.')
```

#### Solution 2 (Calculate 4-Node Cluster):
```python
print(calculate_cluster_sizing(4, 16, 64))
```

#### Solution 3 (Tiny vs Fat Executor Trade-off):
```python
print('1-core loses broadcast sharing; 16-core suffers massive JVM Stop-The-World GC pauses.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Formula**: 5 cores per executor; 10% memory overhead; reserve 1 core/4GB per node for OS.

### Essential Technical Terms
- `Golden Rule`: 3-5 cores per executor.
- `Overhead`: Off-heap buffer.

### Common Engineering Mistakes
- **Fat executors (>32 GB RAM, 16 cores)**: Leads to catastrophic GC freezes.

### Interview Questions on This Topic
- **Q: Why is 5 cores per executor considered the industry best practice?**
  - *A*: It balances JVM Garbage Collection pause times, allows shared broadcast variable caching, and maximizes HDFS I/O throughput.


### More Interview Questions

- **Q: Why not always use the largest possible executors?**
  - *A*: Large executors can fragment the cluster, increase GC pause time, and make a single failure more costly; smaller executors pack better and isolate failures.
- **Q: How do executor cores affect memory per task?**
  - *A*: All concurrent tasks share the executor heap, so more cores per executor generally means less memory available per task.

### What to Study Next
Open 05_spark_connect.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')